In [24]:
import pandas as pd
import pyterrier as pt
import os
from ir_measures import *
pd.set_option('display.max_rows', 500)

In [25]:
def _sens_docs(qrels, run):
    if "sensitivity" in run.columns:
        run = run.drop(columns = ["sensitivity"])
    merged = pd.merge(run, docs, left_on = "doc_id", right_on = "docno")
    return merged.sensitivity.sum()
    
import ir_measures
sens_docs = ir_measures.define_byquery(
    _sens_docs, 
    name="sens_docs")

In [26]:
docs = pd.read_pickle("/nfs/primary/sas_reranker/ohsumed_docs_w_t5base_sensitivity.pkl")
queries = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/queries.pkl")
qrels = pd.read_pickle("/nfs/primary/graph_adaptive_reranking/qrels.pkl")

In [27]:
run_names = []
runs = []
for file in os.listdir("."):
    if "ipynb" not in file and "test" not in file:
        run_names.append(file)
        runs.append(pt.io.read_results(file))

In [28]:
retrieval_results = pt.Experiment(
    runs,
    queries,
    qrels,
    eval_metrics=[nDCG@10, sens_docs@10],
    names = run_names
)

retrieval_results

,name,nDCG@10,sens_docs@10
0,l1reg_add.run,0.520801,1.339623
1,l1reg_multiply.run,0.439655,1.462264
2,noreg_add.run,0.520801,1.339623
3,noreg_multiply.run,0.477414,1.518868
4,cocondenser_distil.run,0.520801,1.339623
5,l1reg_add_negate.run,0.520801,1.339623
6,l1reg_multiply_negate.run,0.439655,1.462264
7,noreg_add_negate.run,0.520801,1.339623
8,noreg_multiply_negate.run,0.477414,1.518868
9,cocondenser_distil_filter.run,0.515262,1.084906


In [14]:
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns

# # Assuming the dataframe is named df
# # Group by 'qid' and plot the scores for each query
# plt.figure(figsize=(12, 6))
# sns.boxplot(x='qid', y='score', data=runs[-2], showfliers=True)
# plt.title('Distribution of Scores per Query')
# plt.xlabel('QID')
# plt.ylabel('Score')
# plt.xticks(rotation=90)
# plt.show()
# # plt.savefig("../plots/pos_rns_neg_nrns_score_distributions.png", dpi = 1080)

In [15]:
# qrels_w_docs = pd.merge(qrels, docs, on = ["docno"])
# qrels_w_docs[['qid', 'docno', 'label', 'sensitivity']]

# combos  = []
# for _, row in qrels_w_docs.iterrows():
#     combo = ""
#     if row.label == 2:
#         combo += "HR "
#     elif row.label == 1:
#         combo += "PR "
#     elif row.label == 0:
#         combo += "NR "
#     else:
#         print("error")
#     if row.sensitivity == 1:
#         combo += "Sens."
#     elif row.sensitivity == 0:
#         combo += "Not Sens."
#     else:
#         print("error")

#     combos.append(combo)

# qrels_w_docs["combo"] = combos
# qrels_w_docs

In [16]:
# # Count occurrences of each 'combo' per 'qid'
# combo_counts = qrels_w_docs.groupby(["qid", "combo"]).size().unstack(fill_value=0)

# # Create subplots: 2 rows, 1 column
# fig, axs = plt.subplots(2, 1, figsize=(10, 12))

# # Plot first stacked bar chart
# combo_counts.iloc[:len(combo_counts)//2].plot(kind="bar", stacked=True, colormap="tab10", ax=axs[0])
# axs[0].set_xlabel("QID")
# axs[0].set_ylabel("Count")
# axs[0].set_title("Relevance / Sensitivity Per QID")
# axs[0].legend(title="Relevance / Sensitivity")
# axs[0].tick_params(axis='x', rotation=0)

# # Plot second stacked bar chart
# combo_counts.iloc[len(combo_counts)//2:].plot(kind="bar", stacked=True, colormap="tab10", ax=axs[1])
# axs[1].set_xlabel("QID")
# axs[1].set_ylabel("Count")
# # axs[1].set_title("Stacked Bar Chart of Combo Counts per QID")
# axs[1].legend(title="Relevance / Sensitivity")
# axs[1].tick_params(axis='x', rotation=0)

# # Adjust layout
# plt.tight_layout()
# plt.savefig("../plots/relevance_sensitivity_per_qid.png", dpi = 1080)
# plt.show()